In [112]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"


In [113]:

model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [114]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [115]:
model_path = "./tinyllama-lora/checkpoint-370"

In [116]:
base_model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float32, low_cpu_mem_usage=True)
non_instruction_model = PeftModel.from_pretrained(base_model, model_path)
non_instruction_model = non_instruction_model.merge_and_unload()
non_instruction_model = non_instruction_model.to(device)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3302.14it/s]


In [85]:
prompt = "The Loop Activity is a type of Activity that "

In [86]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)


In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [117]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

### Instruction:
What is gateway?
### Input:

### Response:



In [119]:
dataset = load_dataset("json", data_files="data/bpmn_instruction_dataset.jsonl", split="train")
dataset = dataset.select(range(10))  # strip to 10 items for fast iteration
dataset


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10
})

In [120]:
dataset['instruction'][0]

'What is BPMN and what is its primary goal?'

In [121]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [122]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [123]:
# Tokenization with Response Masking
def tokenize_and_mask(example):
    text = example["text"]

    # Tokenize full text
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = enc["input_ids"]

    # Find where '### Response:' starts
    response_marker = "### Response:"
    response_start = text.find(response_marker)

    if response_start != -1:
        # Token index where response begins
        response_token_start = len(tokenizer(text[:response_start])["input_ids"])
    else:
        response_token_start = 0  # if marker not found

    # Clone labels and mask out everything before 'Response'
    labels = input_ids.copy()
    labels[:response_token_start] = [-100] * response_token_start

    enc["labels"] = labels
    return enc

In [124]:
dataset = dataset.map(format_example)
dataset[0]

Map: 100%|██████████| 10/10 [00:00<00:00, 203.79 examples/s]


{'instruction': 'What is BPMN and what is its primary goal?',
 'input': '',
 'output': 'BPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. BPMN creates a standardized bridge between business process design and process implementation. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation.',
 'text': '### Instruction:\nWhat is BPMN and what is its primary goal?\n### Input:\n\n### Response:\nBPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business

In [125]:
# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

Map: 100%|██████████| 10/10 [00:00<00:00, 108.81 examples/s]

Tokenization + masking done.


In [126]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [109]:
instruction_model = get_peft_model(non_instruction_model, lora_config)

c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [127]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)


In [128]:
trainer = Trainer(
    model=instruction_model     ,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [101]:
# Save the model — safe_serialization=False avoids the Windows mmap conflict (os error 1224)
instruction_model.save_pretrained("./tinyllama-instruction", safe_serialization=False)
tokenizer.save_pretrained("./tinyllama-instruction")


('./tinyllama-instruction\\tokenizer_config.json',
 './tinyllama-instruction\\tokenizer.json')

In [104]:
# Test generation — use instruction_model directly (already in memory, freshly trained)
instruction_model.eval()

prompt = "### Instruction:\nWhat is gateway?\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2
    )

# Decode only newly generated tokens (after the prompt)
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print("\nModel Output:\n")
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:




In [103]:

questions = [
    "Describe all five Gateway types in BPMN 2.0 and when to use each.",
    "How does a BPMN process get instantiated? List all valid ways.",
    "What are the BPMN 2.0 conformance classes?"
]

In [78]:
for q in questions:
    print("Question:", q)

    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to(device)
    outputs = non_instruction_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.3
    )
    # Decode only the newly generated tokens (after the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.3
    )
    # Decode only the newly generated tokens (after the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("=" * 80, "\n")


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Describe all five Gateway types in BPMN 2.0 and when to use each.

--- Non-instruction model ---


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




--- Instruction-tuned model ---


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...



--- Instruction-tuned model ---


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Question: What are the BPMN 2.0 conformance classes?

--- Non-instruction model ---


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


457 Table 13.6 – Conformant Classes for Business Process Model and Notation, v2.0 The following table lists each class that is considered conforming to BPMN 2.0 with a description of its scope (i.e., whether it covers all possible scenarios or not). In addition, this section provides an overview of each model element

--- Instruction-tuned model ---


